In [7]:
import cv2
import numpy as np

# Load the candidate's face image
candidate_face_path = 'chirag1.jpg'
candidate_face = cv2.imread(candidate_face_path)

if candidate_face is None:
    print(f"Error: Unable to load image from {candidate_face_path}")
    exit()

# Convert the candidate's face image to grayscale
candidate_face_gray = cv2.cvtColor(candidate_face, cv2.COLOR_BGR2GRAY)

# Resize the candidate's face image to a fixed size (e.g., 168x168)
candidate_face_gray = cv2.resize(candidate_face_gray, (168, 168))

# Load the webcam video capture
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert the frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces in the frame
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Iterate over the detected faces
    for (x, y, w, h) in faces:
        # Extract the face ROI
        face_roi = gray[y:y+h, x:x+w]

        # Resize the face ROI to match the size of candidate_face_gray
        face_roi = cv2.resize(face_roi, (168, 168))

        # Calculate the similarity between the candidate's face and the detected face
        result = cv2.matchTemplate(face_roi, candidate_face_gray, cv2.TM_CCOEFF_NORMED)
        min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
        similarity = max_val

        # Calculate the threshold percentage
        threshold_percentage = similarity * 100

        # Check if the threshold percentage is greater than or equal to 80%
        if threshold_percentage >= 7.5:
            status = "MATCHING"
            color = (0, 255, 0)  # Green
        else:
            status = "NOT MATCHING"
            color = (0, 0, 255)  # Red

        # Draw a bounding box around the detected face
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

        # Display the matching status above the box
        cv2.putText(frame, status, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    # Display the output
    cv2.imshow('frame', frame)

    # Exit on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()